
# Parallel Modeling of Mastery and Stability in Aphasia Recovery

## 1. Research Context & Conceptual Motivation

This analysis evaluates two complementary computational frameworks for tracking recovery in individuals with aphasia within the **BEARS (Balancing Effort, Accuracy, and Response Speed)** treatment paradigm (Evans et al., 2021). 

Historically, learning analytics have relied on accuracy-based models to define "mastery." However, clinical evidence in stroke rehabilitation suggests that accuracy alone is insufficient to characterize the "robustness" of a linguistic representation. A patient may achieve high accuracy while still experiencing significant retrieval delays—a state we define as **Fragile Retrieval**.

### Parallel Modeling Paths:
1. **Bayesian Knowledge Tracing (BKT)**: A discrete-state model representing the probability that a specific lexical item has transitioned from an unlearned to a learned state.
2. **Ex-Gaussian RT Modeling**: A continuous-distribution model that characterizes the "tail" of the reaction time distribution ($\tau$), representing retrieval instability and cognitive effort.

**Alignment with BEARS**: The BEARS framework emphasizes the calibration of speed and accuracy. By modeling both mastery and stability, we can identify when a patient has achieved "system calibration"—the optimal balance of fast and accurate retrieval.



## 2. Data Preparation & Statistical Preprocessing

We analyze trial-level data from **Retrieval (No Prime)** naming tasks. This subset provides the most rigorous test of independent lexical access.

### Preprocessing Protocol:
* **Units**: All Reaction Times (RT) are standardized to seconds.
* **Clinical Boundary Filtering**: Trials are filtered to exclude anticipatory responses (< 0.15s) and non-responses (> 30s), following standardized protocols (Evans et al., 2021).
* **Sample Density**: To ensure stable maximum likelihood estimation (MLE) for Ex-Gaussian parameters, sessions are included only if they contain at least 20 correct trials.


In [ ]:

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import exponnorm, pearsonr, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# Set visual standards
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)

def load_and_preprocess(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing dataset: {filepath}")
    
    df = pd.read_csv(filepath)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df.columns = df.columns.str.lower()
    df.rename(columns={'naming1_resp.corr': 'correct', 'naming1_vocal.rt': 'rt'}, inplace=True)
    df['player'] = df['player'].str.lower()
    
    # Unit correction and boundary filtering
    df = df[(df['rt'] > 0.15) & (df['rt'] < 30)]
    
    # Chronological sorting for Markovian modeling
    df = df.sort_values(by=['player', 'session', 'block_number', 'csv_id'])
    return df

df_raw = load_and_preprocess("2. 2019-10-25_treatment_retrieval_noprime.csv")
print(f"Dataset initialized: {len(df_raw)} trials, {df_raw['player'].nunique()} participants.")



## 3. Bayesian Knowledge Tracing (BKT)

BKT assumes a latent "Knowledge" state ($L$) that transitions from 0 to 1 with probability $T$. 

### Formal Parameters (Heuristic Values):
* $P(L_0) = 0.3$: Initial probability of knowledge.
* $P(T) = 0.1$: Probability of learning.
* $P(G) = 0.2$: Guessing probability.
* $P(S) = 0.1$: Slip probability.

*Note: In this implementation, parameters are heuristic and used for illustrative comparative purposes. Future work should involve item-level parameter estimation using Expectation-Maximization.*


In [ ]:

def update_bkt_stable(p_known, correct, p_t, p_g, p_s):
    '''Numerically stable BKT update with epsilon guards.'''
    eps = 1e-8
    if correct:
        p_correct = p_known * (1 - p_s) + (1 - p_known) * p_g
        p_known_post = (p_known * (1 - p_s)) / (p_correct + eps)
    else:
        p_incorrect = p_known * p_s + (1 - p_known) * (1 - p_g)
        p_known_post = (p_known * p_s) / (p_incorrect + eps)
        
    p_known_next = p_known_post + (1 - p_known_post) * p_t
    return np.clip(p_known_next, 0, 1)

def run_bkt(df):
    results = df.copy()
    results['p_mastery'] = 0.0
    
    # Calculate item-level mastery trajectories
    for (p, stim), group in results.groupby(['player', 'stim_text']):
        mastery_path = []
        curr_p = 0.3 # P(L0)
        for val in group['correct']:
            mastery_path.append(curr_p)
            curr_p = update_bkt_stable(curr_p, val, 0.1, 0.2, 0.1)
        results.loc[group.index, 'p_mastery'] = mastery_path
    return results

df_bkt = run_bkt(df_raw)
session_mastery = df_bkt.groupby(['session', 'player'])['p_mastery'].mean().reset_index()



## 4. Ex-Gaussian Modeling of Retrieval Stability

The Ex-Gaussian distribution is the convolution of a Gaussian and an exponential distribution. It is defined by three parameters:
* **$\mu$ (Mu)**: The mean of the Gaussian component (central processing speed).
* **$\sigma$ (Sigma)**: The standard deviation of the Gaussian component.
* **$\tau$ (Tau)**: The mean of the exponential component ($1/\lambda$).

**Interpretation of $\tau$**: In linguistic tasks, $\tau$ represents the magnitude of the right-hand "tail." Larger $\tau$ values indicate frequent, disproportionately long retrieval episodes, which we interpret as markers of retrieval instability or cognitive "blocking."


In [ ]:

def fit_exg_tau(rt_array):
    if len(rt_array) < 20: 
        return np.nan
    try:
        # exponnorm fit: K (shape), loc (mu), scale (sigma)
        # tau = K * scale
        params = exponnorm.fit(rt_array)
        return params[0] * params[2] 
    except:
        return np.nan

rt_results = []
# We aggregate observed accuracy alongside RT metrics for non-circular validation
for (sess, p), group in df_raw.groupby(['session', 'player']):
    correct_trials = group[group['correct'] == 1]['rt']
    tau = fit_exg_tau(correct_trials)
    obs_acc = group['correct'].mean()
    rt_results.append({'session': sess, 'player': p, 'tau': tau, 'observed_accuracy': obs_acc})

session_stability = pd.DataFrame(rt_results).dropna()



## 5. Non-Circular Statistical Validation

To validate the utility of these models, we must avoid circular reasoning (e.g., using $P_L$ to predict $P_L$). 

**Validation Protocol**: We evaluate how well current-session parameters ($P_L$ and $\tau$) predict **Observed Accuracy** in the **subsequent treatment session**.

### Statistical Metrics:
1. **Pearson/Spearman Correlations**: Assessing linear and monotonic associations.
2. **Incremental $R^2$**: Determining the unique variance explained by $\tau$ beyond accuracy alone.


In [ ]:

# Merge BKT Mastery with RT Stability & Observed Accuracy
merged_df = pd.merge(session_mastery, session_stability, on=['session', 'player'])

# Create subsequent session target (Observed Accuracy at Session T+1)
merged_df = merged_df.sort_values(['player', 'session'])
merged_df['next_obs_accuracy'] = merged_df.groupby('player')['observed_accuracy'].shift(-1)
analysis_df = merged_df.dropna()

if len(analysis_df) > 10:
    # 1. Base Model: Accuracy ~ Mastery
    base_model = smf.ols('next_obs_accuracy ~ p_mastery', data=analysis_df).fit()
    # 2. Combined Model: Accuracy ~ Mastery + Tau
    full_model = smf.ols('next_obs_accuracy ~ p_mastery + tau', data=analysis_df).fit()
    
    print("--- Predictive Validation Results ---")
    print(f"Base R-squared (Mastery Only): {base_model.rsquared:.3f}")
    print(f"Full R-squared (Mastery + Tau): {full_model.rsquared:.3f}")
    print(f"Incremental R-squared from Tau: {full_model.rsquared - base_model.rsquared:.3f}")
    print("\n--- Regression Coefficients ---")
    print(full_model.summary().tables[1])
else:
    print(f"Insufficient longitudinal data for predictive validation (N={len(analysis_df)}).")



## 6. The Clinical Personality Map: Heuristic State Classification

The following visualization synthesizes both modeling paths. We utilize heuristic thresholds based on the cohort distribution to categorize recovery states. 

**Heuristics**:
* Mastery Threshold: $P_L = 0.75$ (High versus Low mastery probability).
* Stability Threshold: $\tau = 1.0$ seconds (Stable versus Unstable retrieval).


In [ ]:

def categorize_state(row):
    if row['p_mastery'] >= 0.75 and row['tau'] < 1.0: return 'Stable'
    if row['p_mastery'] >= 0.75 and row['tau'] >= 1.0: return 'Fragile'
    if row['p_mastery'] < 0.75 and row['tau'] < 1.0: return 'Emerging'
    return 'Unlearned'

merged_df['clinical_state'] = merged_df.apply(categorize_state, axis=1)

plt.figure(figsize=(12, 7))
palette = {'Unlearned': '#FF3B00', 'Stable': '#2E7D32', 'Fragile': '#FFA000', 'Emerging': '#2962FF'}

# Points
sns.scatterplot(data=merged_df, x='p_mastery', y='tau', hue='clinical_state', palette=palette, 
                s=150, alpha=0.9, edgecolor='black')

# Threshold markers
plt.axvline(0.75, color='gray', linestyle='--', alpha=0.3)
plt.axhline(1.0, color='gray', linestyle='--', alpha=0.3)

plt.title('Clinical Personality Map: Mastery vs. Instability Trajectories', fontsize=14)
plt.xlabel('BKT Mastery Probability ($P_L$)', fontsize=12)
plt.ylabel('Ex-Gaussian Instability ($\tau$)', fontsize=12)
plt.legend(title='Heuristic State', loc='upper right')
plt.show()

print("Interpretation: The 'Fragile' group represents a critical clinical opportunity. These patients have 'learned' the items (high Mastery) but produce them with high destabilization (high Tau).")



## 7. Model Limitations & Future Directions

This analysis represents an exploratory comparative framework. Several limitations should be noted:
1. **Parameter Estimation**: The BKT parameters used here are fixed heuristics. A proper clinical implementation requires item-level estimation via EM algorithms.
2. **Correct-Only RT**: $\tau$ is estimated only from correct trials. This potentially biases the stability measure toward successful retrieval and ignores the information contained in error latencies.
3. **Thresholding**: The "Clinical States" are defined based on population heuristics. Clinical validation against long-term retention (e.g., 6-month follow-up) is required to define these thresholds scientifically.
4. **Cognitive Attribution**: While $\tau$ correlates with effort and instability, it is a mathematical parameter. Directly attributing it to specific cognitive sub-loops requires additional neural or psycholinguistic benchmarking.
